# 🌾 Smart Farmer — Crop Recommendation System
**Supervised Machine Learning | Decision Trees | Joint Distributions**

Complete pipeline:
1. Load & explore the dataset
2. Visualise joint distributions (CO3)
3. Train Decision Tree & Random Forest (CO4)
4. Evaluate and interpret the model
5. Predict a crop from custom soil values

## 0. Install Dependencies (run once on Colab)

In [ ]:
# Uncomment if running on Google Colab
# !pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

## 1. Imports & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# LOCAL path — change to 'crop_recommendation.csv' if running on Colab
DATA_PATH = '../data/crop_recommendation.csv'

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip().lower() for c in df.columns]
if 'label' in df.columns:
    df.rename(columns={'label': 'crop'}, inplace=True)

print('Shape:', df.shape)
print('Crops:', sorted(df['crop'].unique()))
df.head()

In [ ]:
df.describe().round(2)

## 2. Class Distribution

In [ ]:
plt.figure(figsize=(14, 4))
df['crop'].value_counts().plot(kind='bar', color='#4CAF50', edgecolor='white')
plt.title('Number of Samples per Crop')
plt.xlabel('Crop')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Joint Distributions of Soil Features (CO3)

In [ ]:
# Pairplot — shows joint distributions of N, P, K, pH coloured by crop
top_crops = df['crop'].value_counts().head(6).index
subset = df[df['crop'].isin(top_crops)].sample(600, random_state=42)

g = sns.pairplot(
    subset,
    hue='crop',
    vars=['n', 'p', 'k', 'ph'],
    diag_kind='kde',
    plot_kws={'alpha': 0.5, 's': 18},
    palette='tab10'
)
g.figure.suptitle('Joint Distribution of Soil Features by Crop', y=1.02, fontsize=14)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(
    df[['n', 'p', 'k', 'ph']].corr(),
    annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5
)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Preprocessing & Train/Test Split

In [ ]:
le = LabelEncoder()
df['crop_encoded'] = le.fit_transform(df['crop'])

X = df[['n', 'p', 'k', 'ph']]
y = df['crop_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train samples : {X_train.shape[0]}')
print(f'Test  samples : {X_test.shape[0]}')
print(f'Classes       : {list(le.classes_)}')

## 5. Decision Tree Classifier (CO4 — Supervised Learning)

In [ ]:
dt = DecisionTreeClassifier(max_depth=10, criterion='gini', random_state=42)
dt.fit(X_train, y_train)

dt_acc = accuracy_score(y_test, dt.predict(X_test))
print(f'Decision Tree Accuracy: {dt_acc * 100:.2f}%')

In [ ]:
# Visualise a shallow version of the tree (depth=4 for readability)
dt_vis = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_vis.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(
    dt_vis,
    feature_names=['N', 'P', 'K', 'pH'],
    class_names=le.classes_,
    filled=True, rounded=True, fontsize=8,
    impurity=False, proportion=True, ax=ax
)
plt.title('Decision Tree Structure (depth = 4)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100, max_depth=12, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

rf_acc = accuracy_score(y_test, rf.predict(X_test))
print(f'Random Forest Accuracy: {rf_acc * 100:.2f}%')
print(f'Decision Tree Accuracy: {dt_acc * 100:.2f}%')
print(f'Best model: {"Random Forest" if rf_acc >= dt_acc else "Decision Tree"}')

## 7. Model Evaluation

In [ ]:
best_model = rf if rf_acc >= dt_acc else dt
y_pred = best_model.predict(X_test)

print(f'Overall Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n')
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='YlGn',
    xticklabels=le.classes_, yticklabels=le.classes_,
    linewidths=0.5, ax=ax
)
ax.set_xlabel('Predicted Crop')
ax.set_ylabel('Actual Crop')
ax.set_title('Confusion Matrix — Crop Recommendation')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importances = best_model.feature_importances_
feat_names  = ['N', 'P', 'K', 'pH']
idx = np.argsort(importances)

plt.figure(figsize=(7, 4))
colors = ['#81C784' if i != idx[-1] else '#2E7D32' for i in range(len(feat_names))]
plt.barh(
    [feat_names[i] for i in idx],
    importances[idx],
    color=[colors[i] for i in idx]
)
plt.xlabel('Importance Score')
plt.title('Feature Importance for Crop Prediction')
plt.tight_layout()
plt.show()

## 8. Predict a Crop from Custom Soil Values

In [ ]:
# ── Change these values to your soil test results ──
N_val  = 90
P_val  = 42
K_val  = 43
pH_val = 6.5
# ────────────────────────────────────────────────────

sample = np.array([[N_val, P_val, K_val, pH_val]])
pred   = best_model.predict(sample)[0]
crop   = le.inverse_transform([pred])[0]

print('=' * 40)
print(f'  N  = {N_val}  |  P  = {P_val}')
print(f'  K  = {K_val}  |  pH = {pH_val}')
print('=' * 40)
print(f'  Recommended Crop --> {crop.upper()}')
print('=' * 40)